# Load-shedding risk early warning (Bangladesh)

ML Lab project, Dept. of CSE, University of Asia Pacific

Swagoto Utsab Singha Roy (23101124), Lamiya Zaman (23101132), Jannatul Ferdoushi Islam (23101119)

This notebook runs our project on Kaggle. It gets the code from our GitHub repo, checks the saved models, runs the tests and then goes through the steps one by one: data, baselines, the GRU model, the final ensemble and the lab experiments. At the end there is a small demo where you can pick a date and see the forecast.

**Before running:** turn on Internet in the notebook settings, otherwise the repo can't be cloned (Kaggle asks you to verify your phone number for this). If that doesn't work, download the repo as a ZIP, upload it as a Kaggle dataset and add it to the notebook with *Add Input*. The first code cell will find it.

The settings are in the next cell. With `RETRAIN = False` nothing gets trained, it just uses the models saved in the repo and takes a couple of minutes. With `RETRAIN = True` all the models are trained again, which takes around an hour on CPU. `QUICK = True` makes training much shorter (1 seed, 10 epochs) but the scores will be lower than in our report.

In [ ]:
# settings
REPO_URL = "https://github.com/Spirekharn/loadshed_early_warning.git"
RETRAIN  = False   # True = train all models again, False = use the saved models from the repo
QUICK    = False   # only used when RETRAIN is True: 1 seed and 10 epochs instead of 3 seeds
RUN_LONG = False   # ablation + leave-one-division-out (step 5), this is the slowest part

In [ ]:
import os, sys, glob, shutil, subprocess
from pathlib import Path

def find_project():
    # running on our own laptop from inside the repo
    for p in [Path.cwd(), Path.cwd().parent]:
        if (p / "src" / "config.py").exists():
            return p
    work = Path("/kaggle/working/loadshed-early-warning")
    if (work / "src" / "config.py").exists():      # already cloned earlier in this session
        return work
    # repo added as a Kaggle dataset (input folder is read only so we copy it)
    hits = glob.glob("/kaggle/input/**/src/config.py", recursive=True)
    if hits:
        shutil.copytree(Path(hits[0]).parents[1], work)
        return work
    # otherwise clone it from GitHub
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(work)])
    if r.returncode != 0:
        raise RuntimeError("Could not clone the repo. Turn on Internet in the notebook settings "
                           "(needs a verified phone number) or add the repo as a dataset.")
    return work

ROOT = find_project()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# Kaggle has most of these already, holidays sometimes needs to be installed
try:
    import holidays
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "holidays"], check=True)
import numpy as np, pandas as pd, torch, xgboost, sklearn, shap
print("python", sys.version.split()[0], "| torch", torch.__version__, "| xgboost", xgboost.__version__,
      "| sklearn", sklearn.__version__, "| pandas", pd.__version__)
pd.set_option("display.width", 220, "display.max_columns", 40)
from IPython.display import Image, display, Markdown
FIG, TAB = ROOT / "outputs" / "figures", ROOT / "outputs" / "tables"

def show(*names, width=900):
    for n in names:
        f = FIG / n
        if f.exists(): display(Image(str(f), width=width))
        else: print("missing figure:", n)

def run(script, *args):
    # runs one of our scripts and prints the end of its output
    r = subprocess.run([sys.executable, f"scripts/{script}", *args], capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-3000:] if r.returncode else "")
    if r.returncode: raise RuntimeError(f"{script} failed")

Q = ["--quick"] if QUICK else []

## Step 0: check the saved models
The repo has our trained models in `outputs/models`. This runs them on the test set again and compares with the predictions we saved during training. The difference should be basically zero (around 1e-6).

In [ ]:
run("check_saved_models.py")

## Step 1: leakage tests
Checks that the split is by date, recomputes 300 random labels from the raw data, and scrambles all the data after 2024 to make sure the training data, labels and thresholds don't change. Also checks that no NaN goes into the model.

In [ ]:
r = subprocess.run([sys.executable, "tests/test_pipeline.py"], capture_output=True, text=True); print(r.stdout, r.stderr[-2000:])

## Step 2: data, labels and 14-day windows
Low = no load shedding, Medium = shedding/demand is below the threshold, High = above it. The threshold is the median of the non-zero ratios in the training years only (7.6% for the zones, 1.9% for national).

In [ ]:
run("01_prepare.py")
show("eda_demand_shedding.png", "eda_class_counts.png")
pd.read_csv(TAB / "class_counts_by_year_T+1.csv")

## Step 3: baselines
Persistence (tomorrow is the same as today), a simple rule using BPDB's reserve margin, logistic regression, random forest and XGBoost. The SHAP plot shows which features XGBoost uses most.

In [ ]:
if RETRAIN: run("03_baselines.py")
b = pd.read_csv(TAB / "baselines_test_metrics.csv")
display(b[b.group == "All"][["model", "horizon", "macro_f1", "high_recall", "high_precision", "missed_high", "false_alarm"]].round(3))
show("shap_top_T+1.png")

## Step 4: multimodal GRU
Our main model. One GRU reads the grid data, another reads the weather, and a small MLP takes the context (holidays, gas/coal limits, plants in maintenance). Attention combines the three and there are two outputs, T+1 and T+3. It is trained with 3 seeds and early stopping on validation macro-F1. After that we do temperature scaling and choose the abstain threshold, both on the 2024 validation data.

In [ ]:
if RETRAIN: run("04_train_nn.py", *Q)
display(pd.read_csv(TAB / "nn_test_metrics_mean_std.csv", header=[0, 1], index_col=[0, 1]).round(3))
show("calibration_reliability.png", "confusion_mmgru.png", "modality_attention.png", "abstain_coverage_curve.png", width=700)

## Step 5: modality ablation and leave-one-division-out
This is slow so it only trains when both `RETRAIN` and `RUN_LONG` are True. Otherwise it just shows our saved results.

In [ ]:
if RETRAIN and RUN_LONG: run("05_ablation_lodo.py", *Q)
display(pd.read_csv(TAB / "ablation_modalities.csv")[["model", "horizon", "macro_f1", "high_recall", "missed_high", "false_alarm"]].round(3))
show("ablation_modalities.png", "lodo_T+1.png", width=700)

## Step 6: final ensemble
The final alert mixes the GRU, XGBoost and persistence. The weights are picked on the 2024 validation data, then the models are retrained on 2019-2024 and tested once on 2025-2026. It also makes the latest forecast.

In [ ]:
if RETRAIN: run("09_final_pipeline.py")
fc = pd.read_csv(TAB / "final_comparison.csv")
display(fc[["model", "horizon", "macro_f1", "bal_acc", "high_recall", "high_precision", "missed_high", "false_alarm", "ece", "coverage"]].round(3))
show("final_comparison.png")

## Step 7: lab experiments
The experiments from our lab proposal: (a) PR-AUC for all models, (b) attention vs simple concatenation, (c) one multi-task network vs separate networks for T+1 and T+3, and (d) walk-forward validation, where we train on the earlier years and test on the next year, for 2023 to 2026.

In [ ]:
if RETRAIN: run("10_lab_experiments.py", *Q)
display(pd.read_csv(TAB / "lab_pr_auc.csv")[["model", "horizon", "macro_f1", "pr_auc_macro", "pr_auc_high", "roc_auc_macro"]].round(3))
display(pd.read_csv(TAB / "lab_fusion_task_summary.csv").round(3))
wf = pd.read_csv(TAB / "lab_walk_forward.csv")
display(wf.pivot_table(index=["horizon", "model"], columns="test_year", values="macro_f1").round(3))
show("lab_pr_curves.png", "lab_fusion_task.png", "lab_walk_forward.png", width=800)

## Step 8: forecast for any day
Change `AS_OF` to any date between 2019-12-04 and 2026-10-28. The saved GRU models (3 seeds) look at the 14 days before it and predict T+1 and T+3, and the table also shows what actually happened.

In [ ]:
AS_OF = "2026-07-15"     # change this date
run("08_forecast_next_days.py", "--as-of", AS_OF)
display(pd.read_csv(TAB / f"forecast_{AS_OF}.csv"))
display(Image(str(FIG / f"forecast_{AS_OF}.png"), width=650))

## Latest alert
Forecast from the final ensemble using the newest BPDB report we have.

In [ ]:
f = sorted(glob.glob(str(TAB / "forecast_final_*.csv")))[-1]
display(pd.read_csv(f)); display(Image(f.replace("tables", "figures").replace(".csv", ".png"), width=700))

In [ ]:
# summary of all results + a zip of the outputs folder (it shows up in the Output tab)
run("07_report.py")
shutil.make_archive("/kaggle/working/loadshed_outputs" if Path("/kaggle/working").exists() else str(ROOT / "loadshed_outputs"), "zip", ROOT / "outputs")
display(Markdown((ROOT / "outputs" / "summary.md").read_text(encoding="utf-8")[:6000]))